# SAADE–AUA v4 · дисертаційний експеримент: сценарно-адаптивне навчання

**Дослідницьке питання:** чи підвищує *online scenario-adaptive augmentation* стійкість детекторів синтетичного мовлення порівняно з рівномірною випадковою аугментацією **за однакового бюджету навчання**?

- **E1 Static** — без акустичної аугментації;
- **E2 Random** — 4 категорії × 5 рівнів рівноймовірно;
- **E3 Adaptive** — початкові ваги за помилками E1 на окремому adaptation-fold, потім переоцінювання складності сценаріїв **у процесі навчання** E3;
- **Adaption / Validation / Hidden test** мають різні функції. `validation` не бере участі в підборі ваг E3, порога або епох; `hidden_test` ніколи не оцінюється автоматично;
- **21 сценарій** = 1 чистий запис + 20 акустичних перетворень; аудіо й спектрограми можна слухати / переглядати;
- **Два режими джерел:** `existing_AUA` (пілот із ризиком source/class confounding), `matched_tts` (доступний лише після справжньої генерації й перевірки пар).

❗ **Метод поки не довів перевагу E3.** Підтвердження / спростування — результат незалежного експерименту, а не умова програми.

Не модифікує AUA; не створює файлів у Melnyk_dus. Для будь-якого нового протоколу створюйте нову папку результатів.


## 0. Шляхи, параметри та режими

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys, json, importlib.util
import numpy as np, pandas as pd
from IPython.display import display

PROJECT_ROOT=Path('/home/jovyan/work/audio_real_fake')
DATA_ROOT=PROJECT_ROOT/'AUA'
RESULTS_ROOT=PROJECT_ROOT/'results'
OUT=RESULTS_ROOT/'SAADE_AUA_2026_v4_ADAPTIVE_THESIS'

SOURCE_MODE='existing_AUA'  # 'matched_tts' ONLY if TTS receipts have been generated
TTS_RECEIPTS_CSV=OUT/'V4_00_PIPER_GENERATION_RECEIPTS.csv'
RUN_FULL_TRAINING=False  # Явно встановіть True для повного експерименту
RUN_FAST_SMOKE=False     # True = 1 seed, 1 epoch, MFCC-SVM; лише тест працездатності
RUN_FINAL_HIDDEN_TEST=False  # Заблоковано: інший протокол для одноразового відкриття

SEEDS=(42,123,777)
EPOCHS=15
UPDATE_EVERY=3
MODELS=('spectrogram_cnn','raw_wave_cnn','mfcc_svm')
MAX_PARENT_FILES=900
UNKNOWN_TRAIN_CAP=150
TARGET_SEGMENTS=1200
AUGMENTATION_PROBABILITY=.50
REAL_FRACTION=.50
ADAPT_MAX_PER_CLASS=24

assert DATA_ROOT.is_dir(),f'Набір не знайдено: {DATA_ROOT}'
assert PROJECT_ROOT.is_dir(),f'Папку проєкту не знайдено: {PROJECT_ROOT}'
assert not RUN_FINAL_HIDDEN_TEST, ('Відкриття hidden test дозволено лише після '
  'фіксації протоколу та нового незалежного тесту.')
OUT.mkdir(parents=True,exist_ok=True)
print('AUA:',DATA_ROOT,'\nRESULTS:',OUT,'\nРежим:',SOURCE_MODE)


## 1. Підключення модулів v4 (без конфлікту з v1/v2)

In [ ]:
sys.path.insert(0,str(Path.cwd()))
import saade_v3_core as S
import saade_v3_data as D
import saade_v3_methods as M
import saade_v3_visuals as V
import saade_v4_adaptive as A

for module in (S,D,M,V,A):print(module.__name__,Path(module.__file__).resolve())
assert S.Config.__dataclass_fields__['root'].default.endswith('/audio_real_fake/AUA')
assert all(Path(m.__file__).parent==Path.cwd().resolve() for m in (S,D,M,V,A))
import shutil
assert shutil.which('ffmpeg'), 'Потрібен системний ffmpeg для MP3-сценаріїв'

cfg=S.Config(root=str(DATA_ROOT),out=str(OUT),seed=42,scenario_seed=1001,
    scenario_profile='v2_5level',seconds=4.0,sample_rate=16000,
    max_items=MAX_PARENT_FILES,epochs=EPOCHS,models=MODELS,
    augmentation_probability=AUGMENTATION_PROBABILITY,bootstraps=200)
S.save_json(OUT/'V4_00_CONFIG.json',dict(
    data_root=str(DATA_ROOT),result_root=str(OUT),source_mode=SOURCE_MODE,
    seed_list=SEEDS,epochs=EPOCHS,update_every=UPDATE_EVERY,
    source_class_confounding='UNRESOLVED unless matched_tts fully verified',
    independent_hidden_test='NOT RUN',models=MODELS,scenario_profile=cfg.scenario_profile))
print('✅ Конфігурацію перевірено')

## 2. Метадані AUA та контроль походження

Для DRSSU перевіряються `selection_rank`, `generated`, `source_sha256`; для нових TTS — реєстр `model_name/voice_id`; невідомі голоси не видаються за незалежних мовців.

In [ ]:
catalog=D.build_verified_catalog(cfg)
preflight=(catalog.groupby(['source','label','identity_status'],dropna=False)
         .agg(files=('path','size'),usable=('usable','sum'),groups=('group_id','nunique'))
         .reset_index())
display(preflight)
print('Придатні:',int(catalog.usable.sum()),'із',len(catalog))
print('DRSSU fake:',int(((catalog.source=='DRSSU') & (catalog.label==1)).sum()))


## 3. Group-disjoint train/adaptation/validation/hidden_test

Важливо: 1) групи діляться **до сегментування**; 2) adaptation відбирається **лише з train**; 3) клас балансується в кожній вибірці **випадковим downsampling**, але файли AUA не видаляються.

**Якщо `SOURCE_MODE='existing_AUA'`, це пілот; реальне та синтетичне аудіо походить із різних корпусів.**

In [ ]:
parents,summary=D.prepare_parent_splits(catalog,cfg,
  max_parent_files=MAX_PARENT_FILES,unknown_train_cap=UNKNOWN_TRAIN_CAP,
  policy='exploratory')
print('Первинних записів:',len(parents))
display(summary.head(30))

# За наявності реальних згенерованих пар — використовуємо лише справжні REAL/TTS зв'язки.
if SOURCE_MODE=='matched_tts':
    if not TTS_RECEIPTS_CSV.is_file():
        raise FileNotFoundError('Немає receipts фактично згенерованих TTS: '+str(TTS_RECEIPTS_CSV))
    parents=A.integrate_tts_pairs(parents,TTS_RECEIPTS_CSV,cfg)

# Тексти для майбутньої TTS-генерації, окремо від поточного експерименту.
queue=A.generate_tts_job_queue(parents,cfg,max_real=400)
print('У черзі на TTS після ручної перевірки:',len(queue))
display(queue.head(5))


### 3.1. Окреме adaptation-fold та збалансовані сегменти

In [ ]:
parent_folds=A.create_adaptation_fold(parents,cfg,seed=42,target_fraction=.22)
raw_segments,raw_counts,segment_report=M.expand_nonoverlapping_segments(
    parent_folds,cfg,target=TARGET_SEGMENTS,max_per_parent=30)

if SOURCE_MODE=='matched_tts':
    splits=A.balance_pairs(raw_segments,cfg,seed=42)
    balance_report=splits.groupby(['split','label']).size().rename('n').reset_index()
else:
    splits,balance_report=A.balance_per_partition(raw_segments,cfg,real_pct=REAL_FRACTION,seed=42)

display(balance_report)
source_audit,voice_risks=A.source_and_split_diagnostics(splits,cfg)
display(voice_risks)
assert splits.groupby('group_id').split.nunique().max()==1
assert splits.groupby('parent_file_id').split.nunique().max()==1
print('Фінальна кількість фрагментів:',len(splits))
# Фіксація нової протокольної вибірки; старі експерименти не змінюємо.
splits.to_csv(OUT/'V4_02_LOCKED_EXPERIMENT_SEGMENTS.csv',index=False)
S.save_json(OUT/'V4_02_SPLIT_HASH.json',dict(sha256=S.frozen_hash(splits),
  purpose='freeze before training; never choose E3 parameters from validation/hidden_test',
  hidden_test_opened=False))


## 4. 21 акустичний сценарій: параметри, осцилограми, спектрограми, прослуховування

In [ ]:
scenario_guide=V.draw_scenario_guide(cfg,tag='v4')
# Два записи: один REAL і один FAKE із validation.
examples=V.pick_examples(splits,'validation')
for r in examples:
    print('REAL' if int(r.label)==0 else 'FAKE',r.relative_path)
    V.show_audio_gallery(r,cfg,tag='v4_preview')


### 4.1. Вибір довільного сценарію для прослуховування

Працює в JupyterLab із `ipywidgets`; статичні плеєри вище — без `ipywidgets`.

In [ ]:
interactive=V.make_interactive_player(splits,cfg,'validation')

## 5. E1, E2, E3: рівні умови навчання

У кожному seed кожна модель навчається однакову кількість епох і отримує однакову кількість прикладів. E2 та E3 використовують однакову ймовірність аугментації. В E3 **оновлення ваг здійснюється під час навчання**, використовуючи лише `adaptation`.

Для повного експерименту встановіть `RUN_FULL_TRAINING=True` та `RUN_FAST_SMOKE=False` у початковій комірці. Обидва режими одночасно вмикати не можна.

Не вибирайте найкращий seed за validation. Звітуються всі повтори.

In [ ]:
assert not (RUN_FULL_TRAINING and RUN_FAST_SMOKE)
validation_predictions=None
if RUN_FULL_TRAINING or RUN_FAST_SMOKE:
    used_seeds=SEEDS if RUN_FULL_TRAINING else (42,)
    used_models=MODELS if RUN_FULL_TRAINING else ('mfcc_svm',)
    used_epochs=EPOCHS if RUN_FULL_TRAINING else 1
    validation_predictions,adaptation_predictions=A.evaluate_all_seeds(
        splits,cfg,models=used_models,seeds=used_seeds,epochs=used_epochs,
        update_every=UPDATE_EVERY,adapt_max_per_class=ADAPT_MAX_PER_CLASS)
    print('Усі конфігурації оцінено тільки на validation; hidden_test не відкривався')
else:
    print('Навчання вимкнене. Увімкніть RUN_FULL_TRAINING=True для дисертаційного експерименту.')


## 6. Посценарне тестування: ASRI, EER, AUC, balanced accuracy, F1

In [ ]:
if validation_predictions is not None:
    metric_table,asri_by_seed,paired_deltas,summary=A.metric_tables(validation_predictions,cfg)
    display(asri_by_seed.sort_values(['model','seed','regime']))
    display(paired_deltas)
    display(summary)
    coverage=(validation_predictions.groupby(['seed','model','regime'])
        .agg(scenarios=('category',lambda s:int((validation_predictions.loc[s.index,'category'].astype(str)
              +'::'+validation_predictions.loc[s.index,'level'].astype(str)).nunique())),
             events=('scenario_id','size')).reset_index())
    coverage.to_csv(OUT/'V4_05_SCENARIO_COVERAGE.csv',index=False)
    display(coverage)
    assert coverage.scenarios.eq(21).all(), 'Не всі 21 сценарій присутні'
else:
    print('Результати з’являться після навчання')


## 7. Графіки E3−E2 за всіма 21 сценаріями та трьома seeds

In [ ]:
if validation_predictions is not None:
    import matplotlib.pyplot as plt
    from IPython.display import display
    fig,ax=plt.subplots(figsize=(10,4.8))
    for name,g in paired_deltas.groupby('model'):
        ax.plot(g.seed,g.delta_adaptive_minus_random,marker='o',label=name)
    ax.axhline(0,color='black',linewidth=1)
    ax.set(xlabel='Seed',ylabel='ΔASRI: Adaptive − Random',title='Перевага / відсутність переваги E3 за повтореннями')
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(OUT/'V4_06_ASRI_DELTA_BY_SEED.png',dpi=160);display(fig);plt.close(fig)

    gg=metric_table[metric_table.regime.isin(['random','adaptive'])].copy()
    p=gg.pivot_table(index=['model','seed','category','level'],columns='regime',values='balanced_accuracy').reset_index()
    p['delta_E3_E2']=p['adaptive']-p['random']
    p.to_csv(OUT/'V4_06_SCENARIO_LEVEL_DELTAS.csv',index=False)
    import matplotlib.pyplot as plt
    for name,df in p.groupby('model'):
        fig,ax=plt.subplots(figsize=(10,4))
        for category,sub in df.groupby('category'):
            sub=sub.groupby('level').delta_E3_E2.mean().sort_index()
            ax.plot(sub.index,sub.values,marker='o',label=category)
        ax.axhline(0,color='black',linewidth=1)
        ax.set(xlabel='Рівень інтенсивності',ylabel='Δ Balanced accuracy',
               title=f'{name}: E3 мінус E2 за акустичними сценаріями')
        ax.legend();ax.grid(alpha=.2);fig.tight_layout()
        fig.savefig(OUT/f'V4_06_SCENARIO_DELTA_{name}.png',dpi=150);display(fig);plt.close(fig)


## 8. Статистика, обмеження та перевірка повноти

Показники повторів E1/E2/E3 **не слід трактувати як незалежні спостереження**. Для повноцінної bootstrap-оцінки потрібні **декілька незалежних fake-голосів** на validation; один voice-group не може забезпечити надійний довірчий інтервал узагальнення на невідомі голоси.


In [ ]:
if validation_predictions is not None:
    bootstrap=A.stratified_voice_bootstrap(validation_predictions,cfg,B=300,seed=2026,
                                           min_groups_per_class=3)
    display(bootstrap)
    if (bootstrap.status!='OK').any():
        print('⚠️ Для частини конфігурацій недостатньо незалежних real/fake голосів для bootstrap.')
    # GEE — лише якщо класи представлені кількома незалежними голосами.
    identity_counts=(validation_predictions.groupby('label').group_id.nunique())
    if len(identity_counts)==2 and identity_counts.min()>=3:
        for sd, predictions_one_seed in validation_predictions.groupby('seed'):
            info=M.interaction_statistics(predictions_one_seed,cfg,tag=f'v4_validation_seed{sd}')
            print('GEE / seed',sd,info['status'], 'clusters',info['clusters'])
    else:
        print('GEE пропущено: недостатньо незалежних REAL/FAKE-груп.')


In [ ]:
if validation_predictions is not None:
    risks=voice_risks.set_index('partition')
    n_fake_val=int(risks.loc['validation','independent_fake_groups'])
    print('Незалежних fake-груп у validation:',n_fake_val)
    if n_fake_val<3:
        print('⚠️ СТАТИСТИЧНЕ ОБМЕЖЕННЯ: cluster-bootstrap для узагальнення на нові fake-голоси не виконуємо.')
    else:
        print('✅ Можна провести стратифікований group-bootstrap як окремий наступний аналіз.')
    if SOURCE_MODE!='matched_tts':
        print('⚠️ SOURCE-CLASS CONFOUNDING: підтвердження ефекту E3 на matched ТTS-парах ПОКИ НЕМАЄ.')
    positive=(paired_deltas.delta_adaptive_minus_random>0)
    print('Перемог E3 над E2 за окремими model × seed:',int(positive.sum()),'із',len(positive))
    print('Позитивний середній Δ не означає значущості або причинної переваги без незалежних умов.')
    S.save_json(OUT/'V4_07_RESEARCH_READINESS.json',dict(
        n_fake_validation_groups=n_fake_val,matched_tts_used=SOURCE_MODE=='matched_tts',
        seeds=int(len(used_seeds)),models=list(used_models),
        scenario_profile=cfg.scenario_profile,hidden_test_opened=False,
        claim_ready=bool(SOURCE_MODE=='matched_tts' and n_fake_val>=3 and len(used_seeds)>=3),
        warning='Even a true claim_ready requires verifying capture-channel confounds and independent hypothesis tests.'))


## 9. ZIP зі звітами (без медіа, важких ваг моделей і кешу)

In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
export=OUT.parent/'SAADE_v4_FOR_REVIEW.zip'
files=[p for p in OUT.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv','.json','.png','.txt','.md'} and
       'cache' not in p.parts and 'paired_tts_generated' not in p.parts]
with ZipFile(export,'w',ZIP_DEFLATED) as z:
    for f in files:z.write(f,arcname=str(f.relative_to(OUT)))
print('Готово:',export,'файлів:',len(files),
      'розмір МБ:',round(export.stat().st_size/2**20,2))
print('Набір AUA і папка Melnyk_dus не змінені.')

---
### Умови наукового висновку

Вважати гіпотезу сценарної адаптивності підтриманою **лише якщо** E3 перевершує E2 на незалежній validation за декількома seed і генераторами/голосами, без використання цієї validation для вибору ваг E3; інтервали невизначеності не сумісні з тривіальним ефектом; matched-text дизайн контролює принаймні текстові відмінності. Навіть matched текст сам собою не усуває різницю мікрофонів, каналів, кодеків і мовців.

**Hidden test не оцінюється цим ноутбуком.** Для фінального захищеного тестування потрібна окрема перевірка зафіксованих моделей і джерел, зокрема відсутності витоків голосу й генератора. Експертна оцінка не включена, як було погоджено.